# Phase 4: Controlled Comparative Evaluation

This notebook compares the completed Bicubic, NEDI, FSRCNN, and IMDN results. It does **not** run any super-resolution method again. It validates the saved CSV files, creates combined comparison tables, and generates report-ready graphs.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (start, *start.parents) if (path / 'app').is_dir() and (path / 'results').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the project repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('Repository:', REPO_ROOT)


In [ ]:
from app.evaluation.comparison import (
    build_method_overview,
    generate_phase4_figures,
    load_phase4_summaries,
    validate_detailed_results,
    write_phase4_tables,
)

summary_rows = load_phase4_summaries(REPO_ROOT)
print(f'PASS: {len(summary_rows)} summary rows found (4 methods x 4 datasets x 3 scales).')


## Detailed-result validation

A valid detailed file must contain every expected image once and its averages must match the final summary. Bicubic currently has only its final summary in the repository, so the detailed audit applies to NEDI, FSRCNN, and IMDN.


In [ ]:
detail_checks = {
    'nedi': validate_detailed_results(
        [
            REPO_ROOT / 'results/nedi_x2_all_images_final.csv',
            REPO_ROOT / 'results/nedi_x3_x4_all_images_final.csv',
        ],
        summary_rows,
        'nedi',
    ),
    'fsrcnn': validate_detailed_results(
        [REPO_ROOT / 'results/fsrcnn_all_gpu.csv'], summary_rows, 'fsrcnn'
    ),
    'imdn': validate_detailed_results(
        [REPO_ROOT / 'results/imdn_all_gpu.csv'], summary_rows, 'imdn'
    ),
}
for method, check in detail_checks.items():
    print(f"PASS: {method.upper()} has {check['record_count']} records in {check['group_count']} groups.")


In [ ]:
table_paths = write_phase4_tables(
    summary_rows, REPO_ROOT / 'results/metrics/final', overwrite=True
)
figure_paths = generate_phase4_figures(
    summary_rows, REPO_ROOT / 'results/figures'
)
print('Tables:')
for path in table_paths.values():
    print(' ', path.relative_to(REPO_ROOT))
print('Figures:')
for path in figure_paths:
    print(' ', path.relative_to(REPO_ROOT))


## Main numerical result

The averages below are weighted by image count. This means a 100-image dataset contributes more than Set5, which has only five images. The win columns count how many of the 12 dataset/scale comparisons each method won.


In [ ]:
overview = build_method_overview(summary_rows)
print(f"{'Method':<10} {'PSNR-Y':>10} {'SSIM-Y':>10} {'PSNR wins':>12} {'SSIM wins':>12}")
for row in overview:
    print(
        f"{row['method'].upper():<10} {row['weighted_psnr_y']:>10.4f} "
        f"{row['weighted_ssim_y']:>10.4f} {row['psnr_y_group_wins']:>12} "
        f"{row['ssim_y_group_wins']:>12}"
    )


## Timing interpretation

Do not treat all latency values as a direct hardware-speed competition. Bicubic was timed on Google Colab's default CPU, NEDI was timed on an AWS m7i.2xlarge CPU instance, and FSRCNN and IMDN were timed on an NVIDIA Tesla T4 GPU. The exact Colab CPU model was not recorded. The latency graph therefore records the observed experiment timings and labels the different hardware; it does not claim that the numbers came from one identical device.

## Remaining visual comparison

The numerical part of Phase 4 is reproducible from the CSV files. The remaining part is to place selected reconstructed images from all four methods side by side and discuss edges, textures, smooth areas, and visible artefacts. Those image files must be copied from the experiment output folders before that section can be completed.
